# Lecture audio with Chatterbox (free GPU)

Makes the spoken lectures of Free Exercise Book ahead of time with the open Chatterbox voice
model (MIT licence). See `tools/lecture-audio/README.md` in the repository.

**Kaggle settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **on**.

1. First run with `SAMPLE = True` (a few minutes): listen to the clips under Output.
2. Then set `SAMPLE = False` and use **Save Version → Save & Run All (Commit)**. It runs in the
   background for up to about 11 hours; the clips are kept as this notebook's output.
3. If it stops before everything is made, add the previous output as input
   (**Add Input → Your Work → this notebook**) and run again: finished clips are skipped.
4. Download the output and run `tools/lecture-audio/import.py` on your computer.

In [ ]:
SAMPLE = True          # a few clips from one topic, to listen to first
TOPICS = ""            # e.g. "math.integral.area-length-volume" (empty: every lecture)
EXAGGERATION = 0.45    # expressiveness, 0.25 (flat) to 1.0 (dramatic)
CFG = 0.5              # lower is slower and calmer
MAX_MINUTES = 650      # stop cleanly before Kaggle's 12-hour limit

In [ ]:
!pip install -q chatterbox-tts==0.1.7 "setuptools<81"
!rm -rf repo && git clone -q --depth 1 https://github.com/Kroy665/free_exercise_book.git repo
!nvidia-smi -L; ffmpeg -version | head -1

In [ ]:
# Resume: bring in clips from an earlier run of this notebook added as input.
import glob, os, shutil
os.makedirs("/kaggle/working/clips", exist_ok=True)
for d in glob.glob("/kaggle/input/*/clips"):
    shutil.copytree(d, "/kaggle/working/clips", dirs_exist_ok=True)
print(sum(len(f) for _, _, f in os.walk("/kaggle/working/clips")), "clips already made")

In [ ]:
# One process per GPU, each making every other clip.
import subprocess, torch, time
gpus = max(torch.cuda.device_count(), 1)
args = ["python3", "repo/tools/lecture-audio/generate.py", "repo/content", "/kaggle/working/clips",
        "--exaggeration", str(EXAGGERATION), "--cfg", str(CFG), "--max-minutes", str(MAX_MINUTES)]
if TOPICS: args += ["--topics", TOPICS]
if SAMPLE: args += ["--topics", TOPICS or "math.integral.area-length-volume", "--limit", "2"]
procs = []
for g in range(gpus):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(f"/kaggle/working/gpu{g}.log", "w")
    procs.append(subprocess.Popen(args + ["--shard", f"{g}/{gpus}"], env=env, stdout=log, stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for g in range(gpus):
        lines = open(f"/kaggle/working/gpu{g}.log").read().strip().splitlines()
        print(f"gpu{g}:", lines[-1] if lines else "starting")
for g in range(gpus):
    print(open(f"/kaggle/working/gpu{g}.log").read()[-1500:])
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# Listen to a few clips.
from IPython.display import Audio, display
for f in sorted(glob.glob("/kaggle/working/clips/*/*.mp3"))[:4]:
    print(f); display(Audio(f))